# Fabric defect - Kaggle GPU runs
Settings: Accelerator = GPU (T4/P100), Internet = ON (pretrained weights).
Add the uploaded dataset via 'Add Input'.

In [ ]:
!nvidia-smi | head -n 12
import os, glob, shutil, zipfile
z = glob.glob('/kaggle/input/**/kaggle_upload.zip', recursive=True)
if z:
    zipfile.ZipFile(z[0]).extractall('/kaggle/working/proj')
else:  # dataset uploaded as extracted folder
    root = os.path.dirname(glob.glob('/kaggle/input/**/train.py', recursive=True)[0])
    shutil.copytree(root, '/kaggle/working/proj', dirs_exist_ok=True)
%cd /kaggle/working/proj
!ls
import torch, torchvision; print(torch.__version__, torchvision.__version__, torch.cuda.is_available())

In [ ]:
# 3 CNNs x 3 seeds at 224px. splits.json (fixed) is reused, only the training seed changes.
import subprocess
for seed in (42, 43, 44):
    for m in ('resnet50', 'efficientnet_b0', 'convnext_tiny'):
        subprocess.run(['python', 'train.py', '--model', m, '--data', 'data_cls', '--epochs', '15',
                        '--lr', '1e-4', '--size', '224', '--bs', '32', '--workers', '2',
                        '--seed', str(seed), '--out', f'runs_seed{seed}'], check=True)

In [ ]:
import pandas as pd, glob
df = pd.concat([pd.read_csv(f).assign(seed=f.split('seed')[1].split('/')[0]) for f in glob.glob('runs_seed*/summary.csv')])
print(df.groupby('model')[['test_acc','test_macro_f1','val_best_macro_f1']].agg(['mean','std']).round(4))
!zip -rq /kaggle/working/results.zip runs_seed* -x '*.pt'